# Furax Tutorial

This notebook is a hands-on introduction for new furax.


## 1. What is FURAX?

**Furax** (Framework for Unified and Robust data Analysis with JAX) is a Python library for formulating and solving large-scale linear inverse problems in astrophysics/cosmology, primarily CMB data analysis.

See the accompanying slides for more details and links to other helpful resources: [link](https://docs.google.com/presentation/d/1-uhQcPovfWw1Q3N-U8nTmaAAjSiqMnfYRuLVNgzg_Rk/edit?usp=sharing)

## Installation

### Basic Installation

Install Furax using pip:

```bash
pip install furax
```

### Development Installation

For development or to access the latest features:

```bash
git clone https://github.com/CMBSciPol/furax
cd furax
pip install -e .[dev]
```

**Package structure:**

FURAX has several subpackages:
```
furax/
├── core/        AbstractLinearOperator + concrete operators (Diagonal, Block, Fourier, ...)
├── obs/         Stokes data types, sky landscapes, observation operators (pointing, HWP, ...)
├── mapmaking/   CMB mapmaking pipeline (mapmakers, mapmaking config, noise models, ...)
├── interfaces/  Data adapters: TOAST, sotodlib (SO), LiteBIRD
└── linalg, math Eigendecomposition and quaternion related tools
```

## 2. JAX Essentials

Furax is built on JAX. Understanding a few JAX concepts is essential before working with furax operators.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np

# Allow use of double-precision floats for numerical accuracy.
jax.config.update('jax_enable_x64', True)

print(f'JAX version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

### 2.0 NumPy -> JAX NumPy!

JAX provides a NumPy-inspired interface for convenience. You can simply replace NumPy arrays with JAX arrays in most cases! 

In [ ]:
# Numpy
x_np = np.linspace(0, 1, 4)
x_jax = jnp.linspace(0, 1, 4)

exp_x_np = np.exp(x_np)
exp_x_jax = jnp.exp(x_jax)

f_np = np.fft.fft(x_np)
f_jax = jnp.fft.fft(x_jax)

eig_np = np.linalg.eigvalsh(x_np[:,None] * x_np[None,:])
eig_jax = jnp.linalg.eigvalsh(x_jax[:,None] * x_jax[None,:])

print(f'Not too difficult, is it? {exp_x_np} = {exp_x_jax}')

# Insert your favourite numpy function here

# Replace np with jnp and see what happens

### 2.1 JAX arrays are immutable

Unlike NumPy, JAX arrays **cannot be modified** in place. Use `.at[i].set(value)` to produce an updated copy.

In [ ]:
x_np = np.array([1., 2., 3.])
x_np[0] = 99  # OK in NumPy

x_jax = jnp.array([1., 2., 3.])
# x_jax[0] = 99  # TypeError! JAX arrays are immutable.

# Instead, produce a new array:
x_updated = x_jax.at[0].set(99)
print(f'Original: {x_jax}')   # unchanged
print(f'Updated:  {x_updated}')

# You can use slices too
x_slice_updated = x_jax.at[:2].set(jnp.zeros(2))
print(f'Slice updated:  {x_slice_updated}')

### 2.2 PyTrees

A **PyTree** is any nested Python container (list, tuple, dict, or registered dataclass) whose leaves are JAX arrays. JAX functions like `jit`, `vmap`, and `grad` handle PyTrees natively.

In [ ]:
# Standard PyTree containers
pytree_list   = [jnp.ones(3), jnp.zeros(2), jnp.array(7.)]
pytree_dict   = {'x': jnp.ones(3), 'y': jnp.zeros(2), 'z': jnp.array(7.)}
pytree_nested = {'a': [jnp.ones(3), jnp.zeros(2)], 'b': jnp.array(7.)}

# jax.tree.leaves: flat list of leaves
print('Leaves are identical:')
print('List leaves:', jax.tree.leaves(pytree_list))
print('Dict leaves:', jax.tree.leaves(pytree_dict))
print('Nested leaves:', jax.tree.leaves(pytree_nested))

# jax.tree_util.tree_structure: see the structure
print('\nTree structures are different:')
print('List structure:', jax.tree_util.tree_structure(pytree_list))
print('Dict structure:', jax.tree_util.tree_structure(pytree_dict))
print('Nested structure:', jax.tree_util.tree_structure(pytree_nested))

# jax.tree.map: apply a function element-wise to all leaves
doubled = jax.tree.map(lambda a: 2 * a, pytree_dict)
print('\nDoubled dict:', doubled)

### 2.3 `jax.jit` and tracing

JIT compiles a Python function to XLA on first call, by **tracing** it with abstract (symbolic) values. The output is a compiled binary reused on subsequent calls.

**Caution:** Inside a jitted function, Python conditionals on *traced* values (JAX arrays) raise errors. Use jax numpy functions like `jnp.where` and the low-level function `jax.lax.cond`, or mark the argument as `static`.

In [ ]:
from functools import partial

@jax.jit
def dot_product(a, b):
    return jnp.sum(a * b)

a = jnp.array([1., 2., 3.])
b = jnp.array([4., 5., 6.])
print(f'Dot product: {dot_product(a, b)}')

# Subsequent calls reuse the compiled binary:
print(f'Same result: {dot_product(a * 2, b)}')

# ERROR: cannot branch on traced values
def bad(x, y):
    if y[0] > 0:  # This is traced -- raises error inside jit!
        return x[0]
    return -x[0]

# Use jnp.where(cond, true_values, false_values)
@jax.jit
def good_where(x, y):
    return jnp.where(y[0] > 0, x[0], -x[0])

# Use jax.lax.cond(cond, true_function, false_function, operand)
@jax.jit
def good_cond(x, y):
    return jax.lax.cond(y[0] > 0, lambda y: y, lambda y: -y, x[0])

# print(jax.jit(bad)(a, b))             # Error: traced array used for a python conditional
good_static = jax.jit(bad, static_argnames=('y',))  # Static arguments in JIT must be hashable

print(good_where(a, b))
print(good_where(a, -b))
print(good_cond(a, b))

# print(good_static(a, b))              # Error: static arguments in JIT must be hashable
print(good_static(a, (4., 5., 6.)))     # Tuples are hashable


### 2.4 `jax.vmap`

`vmap` vectorises a function over a batch axis without writing explicit loops. This is useful for distributed computation that GPUs excel at!

In [ ]:
# Create some random vectors and matrices
# Good practice for RNG in JAX: use one main key and 'split' each time a new key is needed.
key = jr.key(67)
key, new_key = jr.split(key)
x = jr.normal(new_key, (4,))
key, new_key = jr.split(key)
y = jr.normal(new_key, (4,))
key, new_key = jr.split(key)
A = jr.normal(new_key, (4, 4))

@jax.jit
def dot_product(x, y):
    return jnp.sum(x * y)

print(f'Dot product: {dot_product(x, y)} = {jnp.dot(x, y)}')

# Vmap over the axis 0 of the first argument, but not second
mv = jax.vmap(dot_product, in_axes=(0, None))
print(f'Matrix-vector product: {mv(A, y)} = {A @ y}')

# Vmap over the axis 0 of all arguments:
row_dot = jax.vmap(dot_product)
print(f'Row dot product: {row_dot(A, A)} = {jnp.sum(A * A, axis=1)}')

### 2.6 `jax.lax.scan` (advanced)

A Python `for` loop inside `jit` **unrolls** — the compiler sees the full loop body repeated. For long loops (e.g., 10,000 TOD samples) this produces enormous compiled code.

`jax.lax.scan` compiles the loop body **once** and runs it repeatedly. The mapmaking pipeline uses scan to accumulate system matrices across observations.

In [ ]:
# Pattern: scan(f, init_carry, xs) -> (final_carry, stacked_outputs)
# f(carry, x_i) -> (new_carry, y_i)

values = jnp.arange(1000, dtype=jnp.float64)

# Accumulate weighted sum with scan:
def step(carry, val):
    return carry + val, None  # no per-step output needed

total, _ = jax.lax.scan(step, 0.0, values)
print(f'Scan sum:   {total}')          # 499500.0
print(f'Direct sum: {jnp.sum(values)}')  # same

# Demonstration: scan inside jit compiles correctly
@jax.jit
def sum_via_scan(xs):
    total, _ = jax.lax.scan(step, 0.0, xs)
    return total

print(f'JIT + scan: {sum_via_scan(values)}')

### 2.7 Common JAX gotchas

| Gotcha | Symptom | Fix |
|--------|---------|-----|
| Forgot `jax_enable_x64` | Silent float32 downcast | Set it before first JAX call |
| In-place mutation | `TypeError` on `x[i] = v` | Use `x.at[i].set(v)` |
| Python branch on traced value | `ConcretizationTypeError` | Use `jax.lax.cond`, or mark arg `static` |
| Shape mismatch after `vmap` | Cryptic error | Check `in_axes` / `out_axes` |
| Unrolled `for` loop inside `jit` | Slow compilation, huge binary | Use `jax.lax.scan` |
| Side effects inside `jit` | Python `print` only runs once | Use `jax.debug.print` |

## 3. FURAX Basics

At the core of furax, we have **linear operators**, which are subclasses of `AbstractLinearOperator`.

Every furax operator is a **frozen dataclass** that is also a **JAX PyTree**. The operator *must*:

1. **Operate on vectors**: implement `mv(x)` for the forward map
2. **Have fixed input structure**: provide `in_structure` at construction (type: `PyTree[jax.ShapeDtypeStruct]`)

Additionally,
- Many operators also **contain additional data** (e.g. diagonal elements of a matrix)
- Operators typically **deduce output structure** `out_structure` automatically via `jax.eval_shape`

### 3.1 Operator basics

In [ ]:
from furax import DiagonalOperator

# DiagonalOperator: element-wise multiplication by a diagonal
diagonal = jnp.array([1., 2., 3., 4., 5., 6.])
op = DiagonalOperator(diagonal, in_structure=jax.ShapeDtypeStruct((6,), jnp.float64))

print('in_structure: ', op.in_structure)   # ShapeDtypeStruct((6,), float64)
print('out_structure:', op.out_structure)  # Same (diagonal is square)

# Apply the operator
x = jnp.ones(6)
print('op(x):  ', op(x))    # [1, 2, 3, 4, 5, 6]
print('op.mv(x):', op.mv(x)) # same; __call__ delegates to mv

# Dense matrix representation (not recommended for large operators)
print('op.as_matrix():\n', op.as_matrix())

# Operators are JAX PyTrees -- their fields are the leaves
print('Operator leaves:', jax.tree.leaves(op))

### 3.2 Operator algebra I

You can treat operators like numpy matrices! You can compose, add, or scale them:

In [ ]:
from furax import IdentityOperator

struct6 = jax.ShapeDtypeStruct((6,), jnp.float64)

A = DiagonalOperator(jnp.array([1., 2., 3., 4., 5., 6.]), in_structure=struct6)
B = DiagonalOperator(jnp.array([6., 5., 4., 3., 2., 1.]), in_structure=struct6)

# Composition: A @ B  (B applied first, then A)
C = A @ B
print(f'(A @ B)(ones): {C(jnp.ones(6))}')   # [6, 10, 12, 12, 10, 6]

# Addition: A + B
D = A + B
print(f'(A + B)(ones): {D(jnp.ones(6))}')   # [7, 7, 7, 7, 7, 7]

# Scalar multiplication: 3 * A
E = 3 * A
print(f'(3*A)(ones):   {E(jnp.ones(6))}')   # [3, 6, 9, 12, 15, 18]

# Compositions are LAZY -- no computation until mv is called:
print(f'\nType of A @ B: {type(C).__name__}')   # CompositionOperator
print(f'Type of A + B: {type(D).__name__}')   # AdditionOperator
print(f'Type of 3 * A: {type(E).__name__}')   # CompositionOperator (Homothety @ A)

### 3.3 Operator algebra II

... and transpose or invert them:

In [ ]:
import lineax as lx
from furax import Config

# .T -- transpose
print('A.T type:', type(A.T).__name__)  # DiagonalOperator (D^T = D for diagonals)

# Verify (A @ B)^T = B^T @ A^T
print('(A@B).T.as_matrix() == B.T @ A.T:', jnp.allclose((A @ B).T.as_matrix(), (B.T @ A.T).as_matrix()))

# .I -- inverse (solves a linear system internally)
A_sq = A.T @ A  # symmetric PSD
x_true = jnp.array([1., 2., 3., 4., 5., 6.])
b = A_sq(x_true)

# Solve A^T A x = b
with Config(solver=lx.CG(rtol=1e-10, atol=1e-10)):
    x_recovered = A_sq.I(b)

print(f'\nRecovery error: {jnp.max(jnp.abs(x_recovered - x_true)):.2e}')

# .as_matrix() -- materialise the dense matrix (only for small operators!)
print(f'\nA as dense matrix:\n{A.as_matrix()}')

### 3.4 Decorated operators 

Operator tags help furax understand it better:

In [ ]:
from furax import tree
from furax.obs.operators import QURotationOperator
from furax.obs.stokes import StokesIQU

# Operator tags describe mathematical properties.
# They are set at class definition via decorators like @symmetric, @orthogonal.

print('DiagonalOperator tags:')
print(f'  is_square:    {A.is_square}')    # True
print(f'  is_symmetric: {A.is_symmetric}') # True
print(f'  is_diagonal:  {A.is_diagonal}')  # True

# Operator tags simplify some operations:
print(f'  A == A.T:     {A == A.T}')         # True

# QURotationOperator is @orthogonal -> T == I (transpose equals inverse)
struct = StokesIQU.structure_for((5,), jnp.float64)
R = QURotationOperator(angles=jnp.full(5, 0.5), in_structure=struct)
print(f'\nQURotationOperator:')
print(f'  is_orthogonal: {R.is_orthogonal}')  # True

# For orthogonal operators, R.T == R.I
x_test = StokesIQU(i=jnp.ones(5), q=jnp.array([0.1]*5), u=jnp.array([0.2]*5))
rotated = R(x_test)
back    = R.T(rotated)  # Uses transpose = inverse for orthogonal
print(f'  Round-trip error: {tree.norm(tree.sub(back, x_test)):.2e}')

### 3.5 Algebraic reduction

FURAX knows how to simplify some operations! Use `.reduce()` to let FURAX run registered rules

In [ ]:
# Algebraic reduction: .reduce() simplifies a composition using registered rules.
# Rules self-register when the module is loaded (no explicit setup needed).

# Example 1: I @ A -> A
I = IdentityOperator(in_structure=struct6)
composed = I @ A
print(f'I @ A type before reduce: {type(composed).__name__}')  # CompositionOperator
print(f'I @ A type after reduce:  {type(composed.reduce()).__name__}')  # DiagonalOperator

# Example 2: R(a) @ R(b) -> R(a+b)  (QURotationRule)
R_a = QURotationOperator(angles=jnp.full(5, 0.3), in_structure=struct)
R_b = QURotationOperator(angles=jnp.full(5, 0.7), in_structure=struct)
RR = R_a @ R_b
print(f'\nR(a) @ R(b) type before reduce: {type(RR).__name__}')  # CompositionOperator
reduced = RR.reduce()
print(f'R(a) @ R(b) type after reduce:  {type(reduced).__name__}')  # QURotationOperator
print(f'Merged angle: {reduced.angles[0]:.4f}')  # 1.0 = 0.3 + 0.7

# Example 3: A.I @ A -> I (InverseBinaryRule)
cancel = A.I @ A
print(f'\nA.I @ A type before reduce: {type(cancel).__name__}')
print(f'A.I @ A type after reduce:  {type(cancel.reduce()).__name__}')  # IdentityOperator

### 3.6 (Advanced) Config - context manager for inverse solvers

Instead of directly passing solver settings at `.I(solver=...)`, you can use furax `Config` as contexts.

In [ ]:
# Config: context manager for solver settings.
# All InverseOperator calls within the context use these settings.

from furax._config import Config

# Default solver is CG with rtol=1e-6, max_steps=500.
print('Default solver:', Config.instance().solver)

# Override for a block:
with Config(solver=lx.CG(rtol=1e-10, atol=0, max_steps=1000),
            solver_throw=True):
    print('Inside context:', Config.instance().solver)
    x_sol = A_sq.I(b)

print('Outside context:', Config.instance().solver)  # back to default
print(f'Solution error: {jnp.max(jnp.abs(x_sol - x_true)):.2e}')

# You can also set the solver inline:
x_sol2 = A_sq.I(solver=lx.CG(rtol=1e-12, atol=0))(b)
print(f'Inline solver error: {jnp.max(jnp.abs(x_sol2 - x_true)):.2e}')

## 4. FURAX General-Purpose Operators

FURAX provides a library of composable operators covering common linear algebra patterns!

### 4.1 Diagonal operator

In [ ]:
from furax import IdentityOperator, DiagonalOperator, BroadcastDiagonalOperator

struct4 = jax.ShapeDtypeStruct((4,), jnp.float64)

# IdentityOperator: I(x) = x. Tagged @diagonal, @orthogonal.
I = IdentityOperator(in_structure=struct4)
print('I.as_matrix():\n', I.as_matrix())

# DiagonalOperator: y_i = d_i * x_i. Tagged @diagonal, @symmetric.
D = DiagonalOperator(jnp.array([1., 2., 4., 8.]), in_structure=struct4)
print('\nD(ones):', D(jnp.ones(4)))    # [1, 2, 4, 8]
print('D.I(ones):', D.I(jnp.ones(4)))  # [1, 0.5, 0.25, 0.125] -- element-wise reciprocal

# BroadcastDiagonalOperator: non-square diagonal with broadcasting.
# values shape (2, 3) + in_structure (3,) + axis_destination=-1 -> output (2, 3).
# Row i of output is values[i] * x.
values = jnp.array([[1., 1., 1.], [2., 1., 0.]])  # shape (2, 3)
bcast_op = BroadcastDiagonalOperator(
    values,
    in_structure=jax.ShapeDtypeStruct((3,), jnp.float64),
    axis_destination=-1,
)
x3 = jnp.array([1., 2., 3.])
print(f'\nBroadcast result shape: {bcast_op(x3).shape}')  # (2, 3)
print(f'Result:\n{bcast_op(x3)}')  # [[1,2,3], [2,2,0]]

### 4.2 Block matrix operations

In [ ]:
from furax import BlockDiagonalOperator, BlockRowOperator, BlockColumnOperator

# BlockDiagonalOperator: applies independent blocks to parts of a PyTree input.
# Input/output share the same PyTree structure as the blocks.

D1 = DiagonalOperator(jnp.array([2., 4.]),   in_structure=jax.ShapeDtypeStruct((2,), jnp.float64))
D2 = DiagonalOperator(jnp.array([1., 3., 5.]), in_structure=jax.ShapeDtypeStruct((3,), jnp.float64))

# List-based blocks
block_diag = BlockDiagonalOperator([D1, D2])
x = [jnp.ones(2), jnp.ones(3)]
result = block_diag(x)
print('BlockDiagonal result:', result)
print('as_matrix():\n', block_diag.as_matrix())

# Dict-based blocks (and Stokes-based -- see Section 7)
D_i = DiagonalOperator(jnp.ones(4),  in_structure=jax.ShapeDtypeStruct((4,), jnp.float64))
D_q = DiagonalOperator(0.5*jnp.ones(4), in_structure=jax.ShapeDtypeStruct((4,), jnp.float64))
D_u = DiagonalOperator(0.5*jnp.ones(4), in_structure=jax.ShapeDtypeStruct((4,), jnp.float64))

stokes_op = BlockDiagonalOperator(StokesIQU(D_i, D_q, D_u))
sky_in = StokesIQU(i=jnp.ones(4), q=jnp.ones(4), u=jnp.ones(4))
print('\nStokes BlockDiagonal result:', stokes_op(sky_in))

In [ ]:
# BlockRowOperator: [A | B | C] -- sums outputs of all blocks.
# Transpose is BlockColumnOperator.

A1 = IdentityOperator(in_structure=jax.ShapeDtypeStruct((2,), jnp.float64))
A2 = 2 * IdentityOperator(in_structure=jax.ShapeDtypeStruct((2,), jnp.float64))
A3 = 3 * IdentityOperator(in_structure=jax.ShapeDtypeStruct((2,), jnp.float64))

row_op = BlockRowOperator([A1, A2, A3])  # All blocks must have same out_structure
print('BlockRow as_matrix():\n', row_op.as_matrix())
# [[1 0 2 0 3 0]
#  [0 1 0 2 0 3]]

x_list = [jnp.array([1., 1.]), jnp.array([1., 1.]), jnp.array([1., 1.])]
print('BlockRow result:', row_op(x_list))  # [6, 6]

# Transpose: BlockRowOperator.T = BlockColumnOperator
# [ A.T ]
# [ B.T ]
# [ C.T ]
col_op = row_op.T
print('BlockColumn type:', type(col_op).__name__)
y = jnp.array([1., 1.])
print('BlockColumn result:', col_op(y))  # [[1,1], [2,2], [3,3]]

In [ ]:
from furax import TreeOperator
from furax.obs.stokes import StokesIQUV

# TreeOperator can represent generalized matrices that takes PyTrees and outputs PyTrees
# The outer PyTree defines rows; inner PyTree (= in_structure) defines columns.
# More memory-efficient than DenseOperator when many entries are zero or shared.

# Example: quarter-wave plate (vertical fast axis)
# [[1,  0,  0,  0],
#  [0,  1,  0,  0],
#  [0,  0,  0, -1],
#  [0,  0,  1,  0]]
qwp = TreeOperator(
    StokesIQUV(
        StokesIQUV(1., 0.,  0.,  0.),  # row I
        StokesIQUV(0., 1.,  0.,  0.),  # row Q
        StokesIQUV(0., 0.,  0., -1.),  # row U
        StokesIQUV(0., 0.,  1.,  0.),  # row V
    ),
    in_structure=StokesIQUV.structure_for((), jnp.float64),
)

print('QWP as_matrix():\n', qwp.as_matrix())

# Apply to a Stokes vector
stokes_in = StokesIQUV(i=1., q=1., u=1., v=1.)
stokes_out = qwp(stokes_in)
print(f'\nQWP(I=1,Q=1,U=1,V=1): I={stokes_out.i:.1f}, Q={stokes_out.q:.1f}, U={stokes_out.u:.1f}, V={stokes_out.v:.1f}')
# Expected: I=1, Q=1, U=-1, V=1  (U and V swapped with sign change)

### 4.3 Indexing operations

Use `IndexOperator` for standard indexing operations, such as slicing an array or reordering.

Note that JAX only works with indexing operations where the output structure is known in advance!

In [ ]:
from furax import IndexOperator

# IndexOperator: extracts elements by advanced indexing.
# y = x[indices]  (forward: gather)
# x = scatter(y)  (transpose: scatter/accumulate)

struct8 = jax.ShapeDtypeStruct((8,), jnp.float64)
indices = jnp.array([0, 2, 5, 7])  # select 4 out of 8 elements

P = IndexOperator(indices, in_structure=struct8)

x = jnp.array([10., 20., 30., 40., 50., 60., 70., 80.])
print('P(x):', P(x))   # [10, 30, 60, 80]  -- selected elements
print('P.T(y):', P.T(jnp.ones(4)))  # scatter 1s into positions [0,2,5,7]

# IndexOperator is the building block of the pointing operator:
# sky_at_pixels = P(sky_map)  -- gather sky values at observed pixels

### 4.4 Asoperator

You can use `asoperator` as a wrapper for anything not yet in FURAX! For example:

- Standard JAX library functions not yet implemented in FURAX
- Simple but specific scripts
- When you need a quick solution


In [ ]:
from furax import asoperator

# asoperator: wraps any JAX function as an AbstractLinearOperator.
# Transpose is computed automatically via jax.linear_transpose.
# Inverse uses the configured solver.

# Example: smoothing operator (circular convolution with a Gaussian kernel)
n = 32
kernel = jnp.exp(-0.5 * (jnp.arange(n) ** 2) / 4.0)  # half-Gaussian kernel
kernel = kernel / jnp.sum(kernel)

def smooth(x):
    return jnp.real(jnp.fft.ifft(jnp.fft.fft(x) * jnp.fft.fft(kernel)))

smooth_op = asoperator(smooth, in_structure=jax.ShapeDtypeStruct((n,), jnp.float64))

x = jnp.zeros(n).at[n // 2].set(1.0)  # impulse
y = smooth_op(x)
print('Smoothed impulse (peak region):', y[n//2 - 3 : n//2 + 4])

# Transpose is available
y_t = smooth_op.T(y)
print('Transpose applied, shape:', y_t.shape)

## 5. Creating Your Own Operator

It's not difficult to create your own operator!

### 5.1 Operator requirements
Every furax operator follows the same pattern:
1. Subclass `AbstractLinearOperator` (becomes a frozen dataclass + JAX PyTree automatically)
2. Declare fields
3. Implement `mv(self, x)`
4. Optionally apply tag decorators (`@symmetric`, `@diagonal`, etc.)

In [ ]:
from furax import AbstractLinearOperator, symmetric, positive_semidefinite
from dataclasses import field

# A symmetric, positive-semidefinite operator: y = a * D * x
# where D is a diagonal and a is a static (compile-time constant) scale.

@positive_semidefinite
@symmetric
class ScaledDiagonalOperator(AbstractLinearOperator):
    """Applies y_i = scale * diagonal_i * x_i.
       Scale and diagonals are assumed to be non-negative. """

    diagonal: jax.Array                              # dynamic: traced by JAX
    scale: float = field(metadata={'static': True})  # static: triggers recompile if changed

    def mv(self, x):
        return self.scale * self.diagonal * x


# Construction: in_structure is required
struct5 = jax.ShapeDtypeStruct((5,), jnp.float64)
op = ScaledDiagonalOperator(
    diagonal=jnp.array([1., 2., 3., 4., 5.]),
    scale=2.0,
    in_structure=struct5,
)

print('op type:', type(op).__name__)
print('is_symmetric:', op.is_symmetric)           # True (from @symmetric)
print('is_positive_semidefinite:', op.is_positive_semidefinite)  # True
print('out_structure:', op.out_structure)          # auto-inferred via eval_shape

x = jnp.ones(5)
print('op(x):', op(x))  # [2, 4, 6, 8, 10]

### 5.2 Static vs Dynamic fields

**Static fields**:
- Values **known** at compile time.
- JAX recompiles when a static field changes.
- Use for: mode flags, shapes, non-array configuration.

**Dynamic fields**:
- Values **unknown** at compile time.
- JAX traces these as symbolic values.
- Use for: arrays, any value that changes between calls.

In [ ]:
import time

# Compile with scale=2.0
op2 = ScaledDiagonalOperator(diagonal=jnp.ones(5), scale=2.0, in_structure=struct5)
jitted_mv = jax.jit(op2.mv)
_ = jitted_mv(jnp.ones(5))  # first call: compiles

# Changing diagonal (dynamic): reuses compiled binary
op3 = ScaledDiagonalOperator(diagonal=jnp.arange(5, dtype=float), scale=2.0, in_structure=struct5)
jitted_mv3 = jax.jit(op3.mv)
_ = jitted_mv3(jnp.ones(5))  # first call with new diagonal: NO recompile
print(f'Dynamic change (no recompile): {jitted_mv3(jnp.ones(5))}')

# Changing scale (static): forces recompile
op4 = ScaledDiagonalOperator(diagonal=jnp.ones(5), scale=3.0, in_structure=struct5)
jitted_mv4 = jax.jit(op4.mv)
_ = jitted_mv4(jnp.ones(5))  # RECOMPILE because scale is a new static value
print(f'Static change (recompiles): {jitted_mv4(jnp.ones(5))}')

In [ ]:
# Verify tag behaviour:
# @symmetric  -> out_structure = in_structure, .transpose() = self
# @positive_semidefinite -> .is_positive_semidefinite = True
# Together: furax picks a good default solver (CG) automatically via lineax

op = ScaledDiagonalOperator(
    diagonal=jnp.array([1., 2., 3., 4., 5.]),
    scale=3.0,
    in_structure=struct5,
)

print('is_square:    ', op.is_square)     # True
print('is_symmetric: ', op.is_symmetric)  # True
print('.T is self:   ', op.T is op)       # True (from @symmetric)

# .I works for symmetric PSD via CG
x_true = jnp.array([1., 2., 3., 4., 5.])
b = op(x_true)
x_recovered = op.I(b)
print(f'\nRecovery error: {jnp.max(jnp.abs(x_recovered - x_true)):.2e}')

# as_matrix: always available (uses fori_loop over basis vectors)
print('\nas_matrix():')
print(op.as_matrix())

## 6. Exercises

These exercises use furax on general (non-astrophysics) linear algebra problems. Solutions use only `furax[dev]`.

### Exercise 1 — Operator composition and the transpose rule

Build two operators:
- `P = IndexOperator` that selects indices `[0, 2, 5]` from an 8-element vector
- `D = DiagonalOperator` with weights `[3., 1., 2.]` applied to the 3-element output of `P`

Form `op = D @ P` and verify algebraically that:
$$
(D \circ P)^\top = P^\top \circ D^\top
$$
by comparing `op.T.as_matrix()` against `P.T.as_matrix() @ D.T.as_matrix()`.

**Expected:** The two matrices should be equal (up to floating-point precision).

In [ ]:
# Exercise 1 -- your solution here
# Hint: IndexOperator(indices, in_structure=...) selects elements;
#       DiagonalOperator(weights, in_structure=...) scales them.

# YOUR CODE HERE


<details><summary>Solution</summary>

```python
struct8 = jax.ShapeDtypeStruct((8,), jnp.float64)
struct3 = jax.ShapeDtypeStruct((3,), jnp.float64)

P = IndexOperator(jnp.array([0, 2, 5]), in_structure=struct8)
D = DiagonalOperator(jnp.array([3., 1., 2.]), in_structure=struct3)

op = D @ P

lhs = op.T.as_matrix()            # (D @ P)^T
rhs = P.T.as_matrix() @ D.T.as_matrix()  # P^T @ D^T

print('LHS:\n', lhs)
print('RHS:\n', rhs)
print('Equal:', jnp.allclose(lhs, rhs))
```

</details>

### Exercise 2 — inverting a matrix with `asoperator`

Construct a random symmetric positive-definite matrix:
$$A = M^\top M + 0.1 I$$
where $M \in \mathbb{R}^{5 \times 8}$ is a random matrix. Wrap $A$ as a furax operator using `asoperator`.

1. Generate a random vector $b \in \mathbb{R}^8$.
2. Solve $Ax = b$ using furax's `.I` with `Config(solver=lx.CG(rtol=1e-10))`.
3. Verify: $\|Ax - b\|_2 < 10^{-8}$.

**Hint:** `asoperator(lambda x: A_matrix @ x, in_structure=...)`

In [ ]:
# Exercise 2 -- your solution here

# YOUR CODE HERE


<details><summary>Solution</summary>

```python
key = jr.key(42)
M = jr.normal(key, (5, 8))
A_matrix = M.T @ M + 0.1 * jnp.eye(8)

struct8 = jax.ShapeDtypeStruct((8,), jnp.float64)
A_op = asoperator(lambda x: A_matrix @ x, in_structure=struct8)

b = jr.normal(jr.key(1), (8,))

with Config(solver=lx.CG(rtol=1e-10, atol=1e-10)):
    x_sol = A_op.I(b)

residual = jnp.linalg.norm(A_op(x_sol) - b)
print(f'||Ax - b|| = {residual:.2e}')  # should be < 1e-8
assert residual < 1e-8, 'Solution not accurate enough'
print('Test passed!')
```

</details>

### Exercise 3 — `BlockDiagonalOperator` on a dict PyTree

Represent the following block-diagonal system using `BlockDiagonalOperator`:
$$
\begin{pmatrix} y_1 \\ y_2 \end{pmatrix}
=
\begin{pmatrix} D_1 & 0 \\ 0 & D_2 \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \end{pmatrix}
$$

where:
- `D1 = DiagonalOperator([1., 2., 3.])` acting on `x1 ∈ R^3`
- `D2 = DiagonalOperator([4., 5.])` acting on `x2 ∈ R^2`

Use a **dict** `{'part1': D1, 'part2': D2}` as the blocks argument.

1. Apply the operator to `x = {'part1': ones(3), 'part2': ones(2)}`.
2. Verify the result matches `{'part1': [1, 2, 3], 'part2': [4, 5]}`.
3. Verify that `.I` applied to the result recovers `x`.

In [ ]:
# Exercise 3 -- your solution here

# YOUR CODE HERE


<details><summary>Solution</summary>

```python
D1 = DiagonalOperator(jnp.array([1., 2., 3.]), in_structure=jax.ShapeDtypeStruct((3,), jnp.float64))
D2 = DiagonalOperator(jnp.array([4., 5.]),     in_structure=jax.ShapeDtypeStruct((2,), jnp.float64))

block_op = BlockDiagonalOperator({'part1': D1, 'part2': D2})

x = {'part1': jnp.ones(3), 'part2': jnp.ones(2)}
y = block_op(x)
print('y:', y)
assert jnp.allclose(y['part1'], jnp.array([1., 2., 3.]))
assert jnp.allclose(y['part2'], jnp.array([4., 5.]))

# Inverse: block diagonal of inverses
x_recovered = block_op.I(y)
print('Recovered x:', x_recovered)
assert jnp.allclose(x_recovered['part1'], jnp.ones(3))
assert jnp.allclose(x_recovered['part2'], jnp.ones(2))
print('Test passed!')
```

</details>

### Exercise 4 — Custom operator: finite differences

Implement a `CirculantDifferenceOperator` on $\mathbb{R}^n$ that computes differences :
$$y_i = x_{i+1} - x_i$$
with periodic boundary: $y_{n-1} = x_0 - x_{n-1}$.

1. Subclass `AbstractLinearOperator`, apply `@square`, implement `mv`.
2. Verify **linearity**: $A(\alpha x + \beta z) = \alpha A(x) + \beta A(z)$ for random $x, z, \alpha, \beta$.
3. Note that the all-ones vector is in the null space of $A$ (why?). Therefore $A^\top A$ is only semi-definite — we cannot invert it directly.
4. Use **Tikhonov regularization** to solve a denoising problem:
   - Generate `x_true = sin(2π k/n)` for some frequency `k`
   - Add noise: `y = A(x_true) + 0.01 * noise`  
   - Recover `x_true` by solving $(A^\top A + \varepsilon I)\hat{x} = A^\top y$ with $\varepsilon = 0.01$
   - Plot `x_true` vs `A(x_hat)` to verify recovery

In [ ]:
# Exercise 4 -- your solution here

# YOUR CODE HERE


<details><summary>Solution</summary>

```python
from furax import square, IdentityOperator

@square
class CirculantDifferenceOperator(AbstractLinearOperator):
    """y[i] = x[(i+1) % n] - x[i] (circulant first difference)."""

    def mv(self, x):
        return jnp.roll(x, -1) - x


n = 64
struct_n = jax.ShapeDtypeStruct((n,), jnp.float64)
A_diff = CirculantDifferenceOperator(in_structure=struct_n)

# 2. Verify linearity
key = jr.key(0)
x_r = jr.normal(key, (n,))
z_r = jr.normal(jr.key(1), (n,))
alpha, beta = 2.3, -1.7
lhs = A_diff(alpha * x_r + beta * z_r)
rhs = alpha * A_diff(x_r) + beta * A_diff(z_r)
print(f'Linearity error: {jnp.max(jnp.abs(lhs - rhs)):.2e}')  # ~1e-15

# 3. Null space: all-ones
ones = jnp.ones(n)
print(f'A(ones) norm: {jnp.linalg.norm(A_diff(ones)):.2e}')  # ~0 (null space)

# 4. Tikhonov denoising
k = 3  # frequency
x_true = jnp.sin(2 * jnp.pi * k * jnp.arange(n) / n)
noise = jr.normal(jr.key(2), (n,))
y = A_diff(x_true) + 0.01 * noise

eps = 0.01
I_op = IdentityOperator(in_structure=struct_n)
AtA_reg = A_diff.T @ A_diff + eps * I_op  # Tikhonov regularizer

with Config(solver=lx.CG(rtol=1e-10, atol=1e-10, max_steps=1000)):
    x_hat = AtA_reg.I(A_diff.T(y))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_true, label='true')
ax.plot(A_diff(x_hat), '--', label='A(x_hat)')
ax.legend()
ax.set_title('Tikhonov-regularized finite-difference inversion')
plt.tight_layout()
plt.show()
print(f'Recovery error: {jnp.linalg.norm(x_hat - x_true):.4f}')
```

</details>

## 7. Summary

### Core patterns

| Pattern | Code |
|---------|------|
| Apply operator | `op(x)` or `op.mv(x)` |
| Compose | `A @ B` (lazy `CompositionOperator`) |
| Transpose | `op.T` |
| Inverse | `op.I(b)` or `op.I(solver=lx.CG(...))(b)` |
| Global solver config | `with Config(solver=...): ...` |
| Algebraic simplify | `(A @ B).reduce()` |
| Debug | `op.as_matrix()` (small operators only) |